In [ ]:
# Import pandas 
import pandas as pd
import numpy as np
import os

In [ ]:
# Defining the path of dataset 
DATA_PATH ="../data/raw"

In [ ]:
# Loading the all dataset 

customers = pd.read_csv(f"{DATA_PATH}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{DATA_PATH}/olist_orders_dataset.csv")
order_items = pd.read_csv(f"{DATA_PATH}/olist_order_items_dataset.csv")
order_payments = pd.read_csv(f"{DATA_PATH}/olist_order_payments_dataset.csv")
order_reviews =  pd.read_csv(f"{DATA_PATH}/olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{DATA_PATH}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{DATA_PATH}/olist_sellers_dataset.csv")
category_translation = pd.read_csv(f"{DATA_PATH}/product_category_name_translation.csv")
geolocation = pd.read_csv(f"{DATA_PATH}/olist_geolocation_dataset.csv")

In [ ]:
# Storing all the above dataset into dictionary

datasets = {
    "customers" : customers ,
    "orders": orders ,
    "order_items" : order_items ,
    'order_payments' : order_payments ,
    "order_reviews" : order_reviews ,
    'products' : products ,
    "sellers" : sellers ,
    "category_translation" : category_translation ,
    "geolocation" : geolocation
}

In [ ]:
# Handling missing values for products ,Orders ,Order Reviews

# Fill missing product dimensions with median values
product_dimensions = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for col in product_dimensions:
    products[col] = products[col].fillna(products[col].median())


# Fill missing product name and description lengths and photo counts
products["product_name_lenght"] = products["product_name_lenght"].fillna(0)
products["product_description_lenght"] = products["product_description_lenght"].fillna(0)
products["product_photos_qty"] = products["product_photos_qty"].fillna(0)

# Get valid category names from the translation dataset
valid_categories = category_translation["product_category_name"].dropna()

# Set missing or unmatched product categories to NaN
products["product_category_name"] = products[
    "product_category_name"
].where(
    products["product_category_name"].isin(valid_categories),
    pd.NA
)

# Keep missing category names as NaN in the translation dataset
category_translation["product_category_name"] = (
    category_translation["product_category_name"].replace("", pd.NA)
)

# Verify missing product categories
print(
    "Missing product categories:",
    products["product_category_name"].isna().sum()
)

# Verify unmatched categories
unmatched_categories = products.loc[
    products["product_category_name"].notna()
    & ~products["product_category_name"].isin(valid_categories),
    "product_category_name"
].unique()

print("Unmatched categories:", unmatched_categories)

# Create separate indicators for missing order dates
orders["delivery_date_missing"] = (
    orders["order_delivered_customer_date"].isnull().astype(int)
)

orders["approval_date_missing"] = (
    orders["order_approved_at"].isnull().astype(int)
)

# Fill missing review comments
order_reviews["review_comment_title"] = (
    order_reviews["review_comment_title"].fillna("No comment title")
)

order_reviews["review_comment_message"] = (
    order_reviews["review_comment_message"].fillna("No comment")
)

# Verify missing dates remain missing
# Check date columns and missing values
print(orders[
    [
        "order_approved_at",
        "order_delivered_customer_date"
    ]
].isnull().sum())


In [ ]:
# Removing Duplicate 

# Removing duplicates from the geolocation dataset

geolocation = geolocation.drop_duplicates()

# Update the dataset dictionary
datasets['geolocation'] = geolocation

print("Duplicate value in geolcation dataset: ",geolocation.duplicated().sum())

In [ ]:
# Correcting the data type

''' Correcting the data type of order_delivered_carrier_date ,order_delivered_customer_date,
order_estimated_delivery_date ,shipping_limit_date ,review_creation_date ,review_answer_timestamp '''

# Orders datatype change
order_datetype_change = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

orders[order_datetype_change] = orders[order_datetype_change].apply(pd.to_datetime)
print(orders[order_datetype_change].dtypes)

# Order Item datatype change
order_item_datatype_change = [
    "shipping_limit_date"
]

order_items[order_item_datatype_change] = order_items[order_item_datatype_change].apply(pd.to_datetime)
print(order_items[order_item_datatype_change].dtypes)

# Order Review datatype change
order_review_datatype_change = [
    "review_creation_date",
    "review_answer_timestamp"
]

order_reviews[order_review_datatype_change] = order_reviews[order_review_datatype_change].apply(pd.to_datetime)
print(order_reviews[order_review_datatype_change].dtypes)


In [ ]:
# Standardize categorical values

categorical_columns = {
    "customers": ["customer_city", "customer_state"],
    "orders": ["order_status"],
    "order_payments": ["payment_type"],
    "products": ["product_category_name"],
    "sellers": ["seller_city", "seller_state"],
    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],
    "geolocation": ["geolocation_city", "geolocation_state"]
}

for dataset_name, columns in categorical_columns.items():
    df = datasets[dataset_name]
    print(f"\n{'=' * 50}")
    print(f"Dataset: {dataset_name}")
    print(f"{'=' * 50}")

    for column in columns:
        df[column] = df[column].str.strip().str.lower()
        print(f"\n{column}:")
        print(df[column].unique())

In [ ]:
# Handling invalid values

# Fix invalid payment installments

order_payments.loc[
    order_payments["payment_installments"] == 0,
    "payment_installments"
] = 1


# Verify the cleaning

print(
    order_payments[
        order_payments["payment_installments"] <= 0
    ]
)



In [ ]:
# Renaming column where required

# We have product_name_lenght and product_description_lenght need renaming

products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
}, inplace=True)

print(products.columns)

In [ ]:
# Save all cleaned datasets in cleaned folder

CLEANED_DATA_PATH = "../data/cleaned"

os.makedirs(CLEANED_DATA_PATH, exist_ok=True)

cleaned_datasets ={
    "customers": customers,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "sellers": sellers,
    "category_translation": category_translation,
    "geolocation": geolocation
}

for name , df in cleaned_datasets.items():
    file_path = os.path.join(CLEANED_DATA_PATH , f'{name}.csv')

    df.to_csv(file_path , index=False)

    print(f'Done {name}.csv')

    print('\nAll cleaned dataset file saved')
